# Práctica: Experimentación/Optimización y Evaluación (Iris)

Objetivos a seguir:

- Clasificar Versicolor vs Virginica usando solo sepal length y sepal width.
- Estandarizar variables con StandardScaler.
- Separar train/test con random_state  =  51.
- Optimizar hiperparámetros solo con train usando remuestreo:
  - LOO, 10-fold, 5x2 y bootstrap.
- Comparar RDA, KDE y k-NN optimizando accuracy, F1 y AUC.
- Evaluar en test con matriz de confusión, métricas avanzadas y curva ROC/AUC.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, StratifiedKFold, LeaveOneOut
from sklearn.preprocessing import StandardScaler

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KernelDensity, KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, confusion_matrix,
    precision_score, recall_score, roc_curve
)

## 1) Carga, filtrado, escalado y split

In [ ]:
iris = load_iris()
X_all = iris.data
y_all = iris.target
names = iris.target_names

mask = (y_all == 1) | (y_all == 2)
X = X_all[mask][:, [0, 1]]  # sepal length, sepal width
y_raw = y_all[mask]

y = np.where(y_raw == 1, 0, 1)

print("Clases: 0 =", names[1], "| 1 =", names[2])
print("Total:", X.shape[0])

# Split train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=51, stratify=y
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print("Train:", X_train_s.shape, "Test:", X_test_s.shape)

## 2) Rejillas de hiperparámetros

In [ ]:
ALPHAS_RDA = [0.0, 0.25, 0.5, 0.75]

KDE_KERNELS = ["gaussian", "tophat"]
KDE_BW = [0.2, 0.5, 0.8]

KNN_K = [1, 3, 5, 7, 9]

METRICS_OPT = ["accuracy", "f1", "auc"]

## 3) Funciones básicas
- Trabajamos con probabilidades para poder calcular AUC/ROC.
- Para KDE calculamos p(x|C0)*P(C0) y p(x|C1)*P(C1) y lo normalizamos.

In [ ]:
def proba_rda(alpha, Xtr, ytr, Xte):
    model = LinearDiscriminantAnalysis(solver="lsqr", shrinkage=alpha)
    model.fit(Xtr, ytr)
    return model.predict_proba(Xte)[:, 1]

def proba_knn(k, Xtr, ytr, Xte):
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(Xtr, ytr)
    return model.predict_proba(Xte)[:, 1]

def proba_kde(kernel, bw, Xtr, ytr, Xte):
    X0 = Xtr[ytr == 0]
    X1 = Xtr[ytr == 1]
    P0 = len(X0) / len(Xtr)
    P1 = len(X1) / len(Xtr)

    kde0 = KernelDensity(kernel=kernel, bandwidth=bw).fit(X0)
    kde1 = KernelDensity(kernel=kernel, bandwidth=bw).fit(X1)

    log0 = kde0.score_samples(Xte)
    log1 = kde1.score_samples(Xte)

    p0 = np.exp(log0) * P0
    p1 = np.exp(log1) * P1
    proba1 = p1 / (p0 + p1 + 1e-12)
    return proba1

def preds_from_proba(proba, thr=0.5):
    return (proba >= thr).astype(int)

def score_metric(y_true, proba, metric_name):
    y_pred = preds_from_proba(proba, 0.5)
    if metric_name == "accuracy":
        return accuracy_score(y_true, y_pred)
    if metric_name == "f1":
        return f1_score(y_true, y_pred)
    if metric_name == "auc":
        return roc_auc_score(y_true, proba)
    raise ValueError("Métrica no soportada")

## 4) Remuestreo en train para optimizar hiperparámetros
Métodos solicitados:
- LOOCV
- 10-fold CV
- Dietterich 5x2 CV
- Bootstrap

In [ ]:
def loocv_scores(get_proba_func, params_list, Xtr, ytr, metric_name):
    loo = LeaveOneOut()
    scores = []
    for params in params_list:
        proba_all = np.zeros(len(ytr))
        for train_idx, val_idx in loo.split(Xtr):
            X_a, X_b = Xtr[train_idx], Xtr[val_idx]
            y_a, y_b = ytr[train_idx], ytr[val_idx]
            proba = get_proba_func(params, X_a, y_a, X_b)
            proba_all[val_idx[0]] = proba[0]
        scores.append(score_metric(ytr, proba_all, metric_name))
    return scores

def kfold_scores(get_proba_func, params_list, Xtr, ytr, metric_name, k=10, seed=51):
    cv = StratifiedKFold(n_splits=k, shuffle=True, random_state=seed)
    scores = []
    for params in params_list:
        proba_all = np.zeros(len(ytr))
        for train_idx, val_idx in cv.split(Xtr, ytr):
            X_a, X_b = Xtr[train_idx], Xtr[val_idx]
            y_a, y_b = ytr[train_idx], ytr[val_idx]
            proba = get_proba_func(params, X_a, y_a, X_b)
            proba_all[val_idx] = proba
        scores.append(score_metric(ytr, proba_all, metric_name))
    return scores

def dietterich_5x2_scores(get_proba_func, params_list, Xtr, ytr, metric_name, seed=51):
    rng = np.random.RandomState(seed)
    n = len(ytr)
    scores = []
    for params in params_list:
        rep_scores = []
        for rep in range(5):
            # split 50/50 estratificado "manual" sencillo
            idx0 = np.where(ytr == 0)[0]
            idx1 = np.where(ytr == 1)[0]
            rng.shuffle(idx0); rng.shuffle(idx1)

            half0 = len(idx0) // 2
            half1 = len(idx1) // 2

            A = np.r_[idx0[:half0], idx1[:half1]]
            B = np.r_[idx0[half0:], idx1[half1:]]

            # 1) entrenar en A, validar en B
            proba1 = get_proba_func(params, Xtr[A], ytr[A], Xtr[B])
            rep_scores.append(score_metric(ytr[B], proba1, metric_name))

            # 2) entrenar en B, validar en A
            proba2 = get_proba_func(params, Xtr[B], ytr[B], Xtr[A])
            rep_scores.append(score_metric(ytr[A], proba2, metric_name))

        scores.append(float(np.mean(rep_scores)))
    return scores

def bootstrap_scores(get_proba_func, params_list, Xtr, ytr, metric_name, B=200, seed=51):
    rng = np.random.RandomState(seed)
    n = len(ytr)
    scores = []
    for params in params_list:
        rep_scores = []
        for _ in range(B):
            idx_boot = rng.randint(0, n, size=n)
            mask_oob = np.ones(n, dtype=bool)
            mask_oob[idx_boot] = False
            idx_oob = np.where(mask_oob)[0]
            if len(idx_oob) == 0:
                continue
            proba = get_proba_func(params, Xtr[idx_boot], ytr[idx_boot], Xtr[idx_oob])
            rep_scores.append(score_metric(ytr[idx_oob], proba, metric_name))
        scores.append(float(np.mean(rep_scores)) if len(rep_scores) > 0 else np.nan)
    return scores

## 5) Optimización: RDA, KDE y k-NN
- Para cada algoritmo:
  - para cada remuestreo (LOO, 10-fold, 5x2, bootstrap)
  - para cada métrica (accuracy, F1, AUC)
  - se elige el hiperparámetro con mejor valor medio

In [ ]:
params_rda = ALPHAS_RDA
params_knn = KNN_K
params_kde = [(ker, bw) for ker in KDE_KERNELS for bw in KDE_BW]

def get_proba_dispatch(algo):
    if algo == "RDA":
        return lambda alpha, Xa, ya, Xb: proba_rda(alpha, Xa, ya, Xb)
    if algo == "kNN":
        return lambda k, Xa, ya, Xb: proba_knn(k, Xa, ya, Xb)
    if algo == "KDE":
        return lambda kb, Xa, ya, Xb: proba_kde(kb[0], kb[1], Xa, ya, Xb)
    raise ValueError("algo")

def optimize_one(algo, resample_name, metric_name, Xtr, ytr, B_boot=200):
    get_proba = get_proba_dispatch(algo)

    if algo == "RDA":
        params_list = params_rda
    elif algo == "kNN":
        params_list = params_knn
    else:
        params_list = params_kde

    if resample_name == "LOO":
        scores = loocv_scores(get_proba, params_list, Xtr, ytr, metric_name)
    elif resample_name == "10fold":
        scores = kfold_scores(get_proba, params_list, Xtr, ytr, metric_name, k=10, seed=51)
    elif resample_name == "5x2":
        scores = dietterich_5x2_scores(get_proba, params_list, Xtr, ytr, metric_name, seed=51)
    elif resample_name == "bootstrap":
        scores = bootstrap_scores(get_proba, params_list, Xtr, ytr, metric_name, B=B_boot, seed=51)
    else:
        raise ValueError("resample_name")

    best_idx = int(np.nanargmax(scores))
    return params_list[best_idx], float(scores[best_idx])

RESAMPLING = ["LOO", "10fold", "5x2", "bootstrap"]
ALGOS = ["RDA", "KDE", "kNN"]

opt_rows = []
for algo in ALGOS:
    for res in RESAMPLING:
        for m in METRICS_OPT:
            best_param, best_score = optimize_one(algo, res, m, X_train_s, y_train, B_boot=200)
            opt_rows.append({
                "algo": algo,
                "resampling": res,
                "metric_opt": m,
                "best_param": best_param,
                "score_train_est": best_score
            })

df_opt = pd.DataFrame(opt_rows)
display(df_opt)

## 6) Gráfica compacta: cómo cambian los hiperparámetros óptimos
- Para RDA: alpha óptimo
- Para k-NN: k óptimo
KDE lo mostramos en tabla (porque kernel+bandwidth es doble).

In [ ]:
df_rda = df_opt[df_opt["algo"] == "RDA"].copy()
df_rda["alpha"] = df_rda["best_param"].astype(float)

plt.figure()
for m in METRICS_OPT:
    tmp = df_rda[df_rda["metric_opt"] == m]
    plt.plot(tmp["resampling"], tmp["alpha"], marker="o", label=m)
plt.title("RDA: alpha óptimo según remuestreo y métrica")
plt.xlabel("remuestreo")
plt.ylabel("alpha")
plt.legend()
plt.show()

# kNN: k óptimo
df_knn = df_opt[df_opt["algo"] == "kNN"].copy()
df_knn["k"] = df_knn["best_param"].astype(int)

plt.figure()
for m in METRICS_OPT:
    tmp = df_knn[df_knn["metric_opt"] == m]
    plt.plot(tmp["resampling"], tmp["k"], marker="o", label=m)
plt.title("k-NN: k óptimo según remuestreo y métrica")
plt.xlabel("remuestreo")
plt.ylabel("k")
plt.legend()
plt.show()

df_kde = df_opt[df_opt["algo"] == "KDE"].copy()
df_kde["kernel"] = df_kde["best_param"].apply(lambda x: x[0])
df_kde["bw"] = df_kde["best_param"].apply(lambda x: x[1])
display(df_kde[["resampling","metric_opt","kernel","bw","score_train_est"]].sort_values(["metric_opt","resampling"]))

## 7) Evaluación final en test
Aquí calculamos para cada modelo óptimo:
- matriz de confusión
- accuracy, precision, recall/sensibilidad, especificidad, F1
- AUC y ROC

In [ ]:
def train_and_predict_proba(algo, best_param, Xtr, ytr, Xte):
    if algo == "RDA":
        return proba_rda(float(best_param), Xtr, ytr, Xte)
    if algo == "kNN":
        return proba_knn(int(best_param), Xtr, ytr, Xte)
    if algo == "KDE":
        ker, bw = best_param
        return proba_kde(ker, bw, Xtr, ytr, Xte)
    raise ValueError("algo")

test_rows = []
roc_store = []  # para graficar luego

for _, row in df_opt.iterrows():
    algo = row["algo"]
    res = row["resampling"]
    met = row["metric_opt"]
    best_param = row["best_param"]

    proba = train_and_predict_proba(algo, best_param, X_train_s, y_train, X_test_s)
    y_pred = preds_from_proba(proba, 0.5)

    cm = confusion_matrix(y_test, y_pred, labels=[0,1])
    tn, fp, fn, tp = cm.ravel()

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred)  # sensibilidad
    spec = tn / (tn + fp + 1e-12)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    auc = roc_auc_score(y_test, proba)

    test_rows.append({
        "algo": algo,
        "resampling": res,
        "metric_opt": met,
        "best_param": best_param,
        "TN": tn, "FP": fp, "FN": fn, "TP": tp,
        "accuracy": acc,
        "precision": prec,
        "recall_sens": rec,
        "specificity": spec,
        "f1": f1,
        "auc": auc
    })

    roc_store.append({
        "algo": algo,
        "resampling": res,
        "metric_opt": met,
        "best_param": best_param,
        "proba": proba
    })

df_test = pd.DataFrame(test_rows)
display(df_test.sort_values(["algo","metric_opt","resampling"]))

## 8) ROC: una figura por algoritmo
Para no sobrecargar, dibujamos por algoritmo las curvas ROC de los modelos optimizados por AUC,
comparando los métodos de remuestreo (LOO, 10fold, 5x2, bootstrap).

In [ ]:
def plot_roc_for_algo(algo_name):
    plt.figure()
    # Solo los modelos optimizados por AUC
    subset = [r for r in roc_store if r["algo"] == algo_name and r["metric_opt"] == "auc"]

    for r in subset:
        fpr, tpr, _ = roc_curve(y_test, r["proba"])
        plt.plot(fpr, tpr, label=f'{r["resampling"]} ({r["best_param"]})')

    plt.plot([0,1], [0,1], linestyle="--", color="gray")
    plt.title(f"ROC en test - {algo_name} (optimizado por AUC)")
    plt.xlabel("FPR")
    plt.ylabel("TPR")
    plt.legend()
    plt.show()

for algo in ALGOS:
    plot_roc_for_algo(algo)

## 9) Bootstrap: estabilidad de accuracy
Se pide ver a partir de qué número de replicaciones la distribución se estabiliza.
Aquí lo hacemos con k-NN usando:
- el k óptimo encontrado por bootstrap optimizando accuracy
- y calculamos la distribución de accuracy OOB para distintos B

In [ ]:
row_knn_boot_acc = df_opt[(df_opt["algo"]=="kNN") & (df_opt["resampling"]=="bootstrap") & (df_opt["metric_opt"]=="accuracy")].iloc[0]
k_star = int(row_knn_boot_acc["best_param"])

def bootstrap_acc_distribution_knn(Xtr, ytr, k, B, seed=51):
    rng = np.random.RandomState(seed)
    n = len(ytr)
    accs = []
    for _ in range(B):
        idx_boot = rng.randint(0, n, size=n)
        mask_oob = np.ones(n, dtype=bool)
        mask_oob[idx_boot] = False
        idx_oob = np.where(mask_oob)[0]
        if len(idx_oob) == 0:
            continue
        proba = proba_knn(k, Xtr[idx_boot], ytr[idx_boot], Xtr[idx_oob])
        accs.append(score_metric(ytr[idx_oob], proba, "accuracy"))
    return np.array(accs)

B_list = [10, 50, 100, 500, 1000]
stats = []

for B in B_list:
    accs = bootstrap_acc_distribution_knn(X_train_s, y_train, k_star, B, seed=51)
    stats.append({"B": B, "mean_acc": float(accs.mean()), "std_acc": float(accs.std(ddof=1))})

df_boot = pd.DataFrame(stats)
display(df_boot)

plt.figure()
plt.plot(df_boot["B"], df_boot["mean_acc"], marker="o")
plt.xscale("log")
plt.title("Bootstrap (kNN): media de accuracy OOB vs B")
plt.xlabel("B (log)")
plt.ylabel("mean accuracy")
plt.show()

accs_100 = bootstrap_acc_distribution_knn(X_train_s, y_train, k_star, 100, seed=51)
accs_1000 = bootstrap_acc_distribution_knn(X_train_s, y_train, k_star, 1000, seed=51)

plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.hist(accs_100, bins=10)
plt.title("Bootstrap accuracy (B=100)")

plt.subplot(1,2,2)
plt.hist(accs_1000, bins=10)
plt.title("Bootstrap accuracy (B=1000)")
plt.tight_layout()
plt.show()

## 10) Comentarios finales

##### Hiperparámetros óptimos:
En RDA, el valor de alpha cambia según la métrica y el remuestreo. Para accuracy y F1, LOO y 10-fold eligen alpha = 0.0, mientras que 5x2 y bootstrap eligen alpha = 0.25. Para AUC se eligen valores más altos: alpha = 0.75 en LOO y 10-fold, alpha = 0.5 en 5x2 y alpha = 0.25 en bootstrap. Esto indica que la métrica AUC favorece más regularización que accuracy o F1.

En KDE, todos los métodos de remuestreo y todas las métricas eligen la misma configuración: kernel gaussian con bandwidth=0.8. Esto muestra que, para estos datos, la configuración más estable es la misma independientemente del método de validación usado.

En k-NN, el valor óptimo de k cambia más. LOO elige k = 5 o k = 7, 10-fold elige k = 3, 5x2 elige k = 7 y bootstrap elige k = 9. Esto refleja que k-NN es más sensible al método de remuestreo, porque cambiar k modifica cuántos vecinos se tienen en cuenta para clasificar cada punto.

##### Diferencias entre remuestreos:
LOO usa casi todos los datos para entrenar en cada repetición, pero puede ser sensible a observaciones individuales. 10-fold es un compromiso más equilibrado. 5x2 usa particiones 50/50 repetidas, por lo que sus resultados pueden variar más al usar menos datos en cada entrenamiento. Bootstrap depende del conjunto OOB y del número de replicaciones; al principio sus estimaciones cambian bastante, pero se estabilizan al aumentar B.

En el ejemplo de bootstrap con k-NN, al aumentar B la media de accuracy OOB pasa de 0.5391 con B = 10 a 0.6121 con B = 1000. La desviación también baja de 0.1418 a 0.0918. A partir de B = 500 la mejora ya es pequeña, por lo que la estimación empieza a estabilizarse.

##### Evaluación en test:
En test, varios modelos alcanzan accuracy = 0.7333. En RDA, las configuraciones con alpha = 0.0 obtienen accuracy = 0.7333, precision = 0.8889, sensibilidad = 0.5333, especificidad = 0.9333, F1 = 0.6667 y AUC = 0.8244. En k-NN, la mejor configuración práctica es k = 7, que obtiene accuracy = 0.7333, precision = 0.7692, sensibilidad = 0.6667, especificidad = 0.8000, F1 = 0.7143 y AUC = 0.7667. Aunque RDA tiene mejor AUC, k-NN con k = 7 consigue mejor equilibrio entre precision y recall, por eso obtiene mayor F1.

##### Accuracy, F1 y AUC:
Accuracy mide el acierto global y es útil cuando las clases están equilibradas. Su limitación es que no distingue qué tipo de error se comete. F1 combina precision y recall, por lo que es útil cuando interesa equilibrar falsos positivos y falsos negativos. AUC mide la capacidad de separar clases al variar el umbral; su ventaja es que no depende de un único umbral, pero puede seleccionar modelos que luego no dan el mejor accuracy o F1 con umbral 0.5.